# SymbolicTensors — Manifold, tangent bundle and frame bundle

This notebook explains the structures defined with @def_manifold macro and the main design of the package

---
## 1. Loading package

In [1]:
using SymbolicTensors

---
## 2. Manifolds and def_manifold macro
---

### a. Manifold type and definition

#### Documentation access
- `@doc Manifold` — shows the docstring directly (julia macro)
- `?Manifold` — same, with search results prepended (Jupyter help mode)

In [2]:
@doc Manifold

```julia
Manifold
```

Struct representing a registered differentiable manifold. Instances are created by [`@def_manifold`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
M.name              # :M
M.dim               # 4
M.tangent_bundle    # :tangentM
M.cotangent_bundle  # :cotangentM
M.vbundles          # [:tangentM, :cotangentM]
```

### Fields

  * `name`             : manifold name, e.g. `:M`
  * `dim`              : dimension (concrete `Int` or symbolic `Symbol`)
  * `tangent_bundle`   : name of the tangent bundle, e.g. `:tangentM`
  * `cotangent_bundle` : name of the cotangent (dual) bundle, e.g. `:cotangentM`
  * `vbundles`         : names of all vector bundles over this manifold


In [3]:
?@def_manifold

```julia
@def_manifold <name> dim coord_indices frame_indices [kwargs...]
```

Define a new manifold and automatically create its tangent and cotangent bundles, coordinate frames, and moving frames.

Both index lists are **required**. Each list should have at least 4 symbols (a warning is issued if fewer).

`dim` accepts a concrete positive integer or a bare symbol (e.g. `d`) for parametric / general-dimension calculations where the dimension is not fixed at definition time.

### Bindings in the caller's scope

  * `<name>`            → [`Manifold`](@ref) instance
  * `tangent<name>`     → [`VBundle`](@ref) (`isref = true`)
  * `cotangent<name>`   → [`VBundle`](@ref) (`isref = false`)
  * `frame<name>`       → [`FrameBundle`](@ref) (moving frame bundle)
  * `coframe<name>`     → [`FrameBundle`](@ref) (moving coframe bundle)
  * coordinate frame bindings `cf_<name>`, `ccf_<name>` (default; display as `∂`, `dx`)
  * moving frame bindings `mf_<name>`, `mcf_<name>` (default; display as `e`, `θ`)
  * each symbol in `coord_indices` → [`CoordinateIndex`](@ref) (contravariant)
  * each symbol in `frame_indices` → [`FrameIndex`](@ref) (contravariant)

### Index variance

Coordinate indices:

```julia
a1          # CoordinateIndex(:a1, :tangentM)   — contravariant
-a1         # CoordinateIndex(:a1, :cotangentM) — covariant
```

Frame indices:

```julia
A1          # FrameIndex(:A1, :tangentM)   — contravariant
-A1         # FrameIndex(:A1, :cotangentM) — covariant
```

### Keyword arguments

| Keyword    | Default                                          | Description                                                                             |
|:---------- |:------------------------------------------------ |:--------------------------------------------------------------------------------------- |
| `frames`   | `[cf_<name>, ccf_<name>, mf_<name>, mcf_<name>]` | Caller-scope binding symbols (coord frame, coord coframe, moving frame, moving coframe) |
| `print_as` | `["∂", "dx", "e", "θ"]`                          | Display label strings for the four bases                                                |

All four entries in each vector must be distinct. Reusing a binding already registered for another manifold triggers a warning.

### Examples

```julia
# Minimal — default bindings cf_M, ccf_M, mf_M, mcf_M; display ∂, dx, e, θ
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
# Binds: M, tangentM, cotangentM, frameM, coframeM,
#        cf_M, ccf_M, mf_M, mcf_M,
#        a1..a4 (CoordinateIndex), A1..A4 (FrameIndex)
ccf_M[a1]   # displays as dx[a1]

# Parametric dimension
@def_manifold M d [a1, a2, a3, a4] [A1, A2, A3, A4]

# Custom bindings and display labels
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4] \
    frames=[:cfM, :ccfM, :mfM, :mcfM] \
    print_as=["∂", "dx", "e", "θ"]
```


In [4]:
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]

Defined manifold M of dimension 4
Defined coordinate frame ∂ (binding :cf_M) on tangentM and coordinate coframe dx (binding :ccf_M) on cotangentM
Defined frame bundle frameM (moving frame e, binding :mf_M) and coframe bundle coframeM (moving coframe θ, binding :mcf_M) over M


#### The different output for a struct like the manifold M 

In [5]:
# Jupyter output (MIME text/html)
M

Dimension,4
Tangent Bundle,tangentM
Cotangent Bundle,cotangentM
All VBundles,"tangentM, cotangentM"


In [6]:
# Built-in repr of julia lang: representative string output ( string of MIME io default)
repr(M)

"Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])"

In [7]:
Meta.parse(Out[6])

:(Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM]))

In [8]:
eval(:(Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])))

Dimension,4
Tangent Bundle,tangentM
Cotangent Bundle,cotangentM
All VBundles,"tangentM, cotangentM"


In [9]:
# More efficient construction of Expr 
expr_form(M)

:(SymbolicTensors.Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM]))

In [10]:
@doc expr_form

```julia
expr_form(x)

Return an expression that reconstructs `x` via its constructor.
Useful for debugging and development: copy-paste the result to recreate objects.
```

#### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
expr_form(a1)
# :(CoordinateIndex(:a1, :tangentM))
expr_form(M)
# :(Manifold(...))  # all field values inlined
```


In [11]:
# dot access to a Manifold properties 
(M.name, M.dim, M.tangent_bundle, M.cotangent_bundle, M.vbundles)

(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])

In [12]:
# Manifold registry
(_MANIFOLDS,M==_MANIFOLDS[:M])

(Dict{Symbol, Manifold}(:M => Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])), true)

In [13]:
(expr_form(M),typeof(expr_form(M)))

(:(SymbolicTensors.Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])), Expr)

In [14]:
typeof(expr_form(M))

Expr

In [15]:
(expr_form(M).head,expr_form(M).args)

(:call, Any[:(SymbolicTensors.Manifold), :(:M), 4, :(:tangentM), :(:cotangentM), [:tangentM, :cotangentM]])

In [16]:
Manifold(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])

Dimension,4
Tangent Bundle,tangentM
Cotangent Bundle,cotangentM
All VBundles,"tangentM, cotangentM"


### b. (co)Tangent bundle, (co)frames

In [17]:
@doc VBundle

```julia
VBundle
```

Struct representing a registered vector bundle. Instances are created by [`@def_manifold`](@ref) for the tangent and cotangent bundles, and by [`@def_vbundle`](@ref) for custom bundles. Bound to variables in the caller's scope (e.g. `tangentM`, `cotangentM`).

Provides dot access to all metadata:

```julia
tangentM.name             # :tangentM
tangentM.manifold         # :M
tangentM.dim              # 4
tangentM.isref            # true
tangentM.dual             # :cotangentM
tangentM.coordinate_indices  # [CoordinateIndex(:a1, :tangentM), ...]
tangentM.frame_indices       # [FrameIndex(:A1, :tangentM), ...]
tangentM.bases            # [Basis(:cf_M, :tangentM, :coordinate, "∂"),
                          #  Basis(:mf_M, :tangentM, :frame, "e")]
```

### Fields

  * `name`               : bundle name, e.g. `:tangentM`
  * `manifold`           : base manifold name, e.g. `:M`
  * `dim`                : fibre dimension
  * `isref`              : true if vbundle was the input to [`@def_vbundle`](@ref), false if it is the dual.                        true -> (ex: tangentM), false -> (ex: cotangentM). Authoritative for variance via                        [`is_up`](@ref) / [`is_down`](@ref).
  * `dual`               : name of the paired dual bundle
  * `coordinate_indices` : [`CoordinateIndex`](@ref) objects for the coordinate                        chart; nonempty for tangent/cotangent bundles
  * `frame_indices`      : [`FrameIndex`](@ref) objects for the fibre frame;                        populated by [`@def_manifold`](@ref) and                        [`@def_vbundle`](@ref)
  * `bases`              : [`Basis`](@ref) objects for the coordinate and frame bases


#### (co)tangent space 

In [18]:
tangentM

VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4, bases=[Basis(:cf_M, :tangentM, :coordinate, "∂"), Basis(:mf_M, :tangentM, :frame, "e")])

In [19]:
(tangentM.name,tangentM.manifold,tangentM.dim,tangentM.isref
,tangentM.dual,tangentM.coordinate_indices,tangentM.frame_indices,tangentM.bases)

(:tangentM, :M, 4, true, :cotangentM, CoordinateIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :tangentM), CoordinateIndex(:a3, :tangentM), CoordinateIndex(:a4, :tangentM)], FrameIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)], Basis[Basis(:cf_M, :tangentM, :coordinate, "∂"), Basis(:mf_M, :tangentM, :frame, "e")])

In [20]:
repr(tangentM)

"VBundle(:tangentM, :M, 4, true, :cotangentM, CoordinateIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :tangentM), CoordinateIndex(:a3, :tangentM), CoordinateIndex(:a4, :tangentM)], FrameIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)])"

In [21]:
expr_form(tangentM)

:(SymbolicTensors.VBundle(:tangentM, :M, 4, true, :cotangentM, CoordinateIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :tangentM), CoordinateIndex(:a3, :tangentM), CoordinateIndex(:a4, :tangentM)], FrameIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)]))

In [22]:
tangentM.bases

2-element Vector{Basis}:
 ∂
 e

In [23]:
repr(tangentM.bases)

"Basis[Basis(:cf_M, :tangentM, :coordinate, \"∂\"), Basis(:mf_M, :tangentM, :frame, \"e\")]"

In [24]:
repr(tangentM.frame_indices)

"FrameIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)]"

In [25]:
FrameIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)]==
[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM), FrameIndex(:A3, :tangentM), FrameIndex(:A4, :tangentM)]

true

#### Basis and BasisElement

In [26]:
@doc Basis

```julia
Basis
```

A named frame for a vector bundle, of a given type (`:coordinate` or `:frame`). Instances are created by [`@def_manifold`](@ref) (coordinate and frame) or standalone [`@def_frame_bundle`](@ref) (frame only for custom bundles).

### Fields

  * `name`     : binding symbol in the caller, e.g. `:cf_M`, `:ccf_M`
  * `vbundle`  : the bundle this frame is for, e.g. `:cotangentM`
  * `type`     : `:coordinate` (coordinate-induced) or `:frame` (arbitrary local frame)
  * `print_as` : display label string, e.g. `"dx"`, `"∂"`, `"e"`, `"θ"`

Index with the **binding** (`ccf_M[a1]`); REPL compact `show` uses `print_as`:

```julia
ccf_M[a1]    # a1 ∈ tangentM  → displays as dx[a1]
mcf_M[-A1]   # -A1 ∈ cotangentM → displays as θ[A1]
```


In [27]:
# Frames and coFrames 
(cf_M[-a1], ccf_M[a1], mf_M[-A1], mcf_M[A1])

(BasisElement(Basis(:cf_M, :tangentM, :coordinate, "∂"), CoordinateIndex(:a1, :cotangentM)), BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a1, :tangentM)), BasisElement(Basis(:mf_M, :tangentM, :frame, "e"), FrameIndex(:A1, :cotangentM)), BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A1, :tangentM)))

In [28]:
cf_M

∂

In [29]:
expr_form(cf_M)

:(SymbolicTensors.Basis(:cf_M, :tangentM, :coordinate, "∂"))

In [30]:
SymbolicTensors.Basis(:cf_M, :tangentM, :coordinate, "∂")

∂

In [31]:
_BASES

Dict{Tuple{Symbol, Symbol}, Basis} with 4 entries:
  (:tangentM, :frame)        => Basis(:mf_M, :tangentM, :frame, "e")
  (:cotangentM, :frame)      => Basis(:mcf_M, :cotangentM, :frame, "θ")
  (:tangentM, :coordinate)   => Basis(:cf_M, :tangentM, :coordinate, "∂")
  (:cotangentM, :coordinate) => Basis(:ccf_M, :cotangentM, :coordinate, "dx")

In [32]:
tangentM.bases

2-element Vector{Basis}:
 ∂
 e

In [33]:
repr(tangentM.bases)

"Basis[Basis(:cf_M, :tangentM, :coordinate, \"∂\"), Basis(:mf_M, :tangentM, :frame, \"e\")]"

In [34]:
cotangentM.bases

2-element Vector{Basis}:
 dx
 θ

In [35]:
repr(cotangentM.bases)

"Basis[Basis(:ccf_M, :cotangentM, :coordinate, \"dx\"), Basis(:mcf_M, :cotangentM, :frame, \"θ\")]"

In [36]:
@doc BasisElement

```julia
BasisElement
```

A single element of a basis (coordinate or frame), constructed by `getindex` on a [`Basis`](@ref).

### Fields

  * `basis` : the [`Basis`](@ref) this element belongs to
  * `index` : the [`AbstractIndex`](@ref) labeling this element;           its vbundle is the **dual** of `basis.vbundle`

    ccf*M[a1] → BasisElement(Basis(:ccf*M, :cotangentM, :coordinate, "dx"), ...)   mcf*M[A1] → BasisElement(Basis(:mcf*M, :cotangentM, :frame, "θ"), ...)   cf*M[-a1] → BasisElement(Basis(:cf*M, :tangentM, :coordinate, "∂"), ...)   mf*M[-A1] → BasisElement(Basis(:mf*M, :tangentM, :frame, "e"), ...)


In [37]:
(ccf_M[a1],cf_M[-a1])

(BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a1, :tangentM)), BasisElement(Basis(:cf_M, :tangentM, :coordinate, "∂"), CoordinateIndex(:a1, :cotangentM)))

In [38]:
(mcf_M[A1],mf_M[-A1])

(BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A1, :tangentM)), BasisElement(Basis(:mf_M, :tangentM, :frame, "e"), FrameIndex(:A1, :cotangentM)))

##### Error handling 

In [39]:
ccf_M[-a1]

LoadError: BasisElement: index vbundle :cotangentM is not the dual of basis vbundle :cotangentM. Expected index from :tangentM. Index the basis binding :ccf_M (prints as dx).

In [40]:
ccf_M[A1]

LoadError: BasisElement: coordinate basis dx requires a CoordinateIndex (e.g. a1 or -a1); got FrameIndex.

#### Change of basis coordinate <-> moving

---
## 3. Tensors
---

### a. Type, definition and tensor components

In [41]:
@doc Tensor

```julia
Tensor
```

A registered abstract tensor. Instances are created by [`@def_tensor`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
T.manifold        # :M  (Symbol key into _MANIFOLDS)
T.slots           # [:cotangentM, :cotangentM]  — vbundle per slot
T.symmetries      # [SlotSymmetry(n=2, order=2)]
T.is_traceless    # false
T.known_traces    # Any[]  (populated later)
T.print_as        # "T"
T.metric          # :g or nothing
T.rank            # 2      (derived — length of slots)
T.vbundle         # :tangentM  (derived — vbundle of reference from slots)
```

### Fields

  * `manifold`      : name of the base manifold, key into `_MANIFOLDS`
  * `slots`         : vbundle symbol per slot, encoding variance.                   `:cotangentM` → covariant, `:tangentM` → contravariant.
  * `symmetries`    : `Vector{`[`SlotSymmetry`](@ref)`}` — list of permutation                   groups acting on slot positions `1:rank`.
  * `is_traceless`  : if `true`, any self-contraction of this tensor gives zero                   (e.g. the Weyl tensor).
  * `known_traces`  : user-declared trace values, e.g. `g[a,-a] = dim`.                   Format TBD — stored as `Any[]` until contraction is                   implemented.
  * `print_as`      : display label string. Controls how the tensor appears in                   `show` and (later) LaTeX output (same convention as                   [`Basis`](@ref) `print_as` in [`@def_manifold`](@ref)).
  * `metric`        : name of the metric tensor associated with this tensor,                   a key into `_METRICS`, or `nothing` if no metric                   has been assigned. Required for raising/lowering indices.

### Reference vbundle invariant

All slots must derive from the same vbundle of reference. For slot `vb`:

  * if `_VBUNDLES[vb].isref`, the vbundle of reference is `vb` itself
  * if `!_VBUNDLES[vb].isref`, the vbundle of reference is `_VBUNDLES[vb].dual`

This invariant is enforced by [`@def_tensor`](@ref) and is accessible via the derived property `T.vbundle`.


In [42]:
@doc @def_tensor

```julia
@def_tensor name [vbundle1, vbundle2, ...] [symmetries=...] [traceless=...] [print_as="..."] [metric=...]
```

Define a new abstract tensor and bind it to `name` in the caller's scope.

### Slot syntax

Specify slots directly as VBundle symbols. The manifold is automatically deduced from the first VBundle's `manifold` field.

  * `:tangentM` → contravariant (upper) slot
  * `:cotangentM` → covariant (lower) slot
  * Any user-defined VBundle from `@def_vbundle`

All slots must derive from the **same vbundle of reference**. A vbundle of reference is any vbundle `v` with `v.isref = true`. For a slot `vb`: if `vb.isref` is `true`, its vbundle of reference is `vb`; if `false`, its vbundle of reference is `vb`'s dual. Mixing slots from distinct vbundles of reference is an error — such objects are not tensors in the strict mathematical sense.

### Registry collision rules

1. Same name, same vbundle of reference, same slots → **warn and redefine**
2. Same name, same vbundle of reference, different slots → **error**
3. Same name, different vbundle of reference → **error**

Rule 3 ensures that a variable name in scope maps to exactly one mathematical object with no ambiguity.

### Keyword arguments (all optional)

  * `symmetries`  : a [`SlotSymmetry`](@ref) or `Vector{SlotSymmetry}` describing                 the slot permutation symmetry group(s). Defaults to                 `[no_symmetry(rank)]`.
  * `traceless`   : `true` if any self-contraction of this tensor is zero                 (e.g. Weyl tensor). Defaults to `false`.
  * `print_as`    : display label string. Defaults to `string(name)`.                 Example: `print_as="R"` or `print_as=:R` (symbol sugar).
  * `metric`      : name of the metric tensor to associate with this tensor.                 Omitting this keyword triggers automatic resolution on the                 tensor's vbundle of reference:                 - one metric on that vbundle → assigned silently                 - multiple metrics → `@warn`, first sorted name is assigned                 - no metric → `@warn`, `nothing` assigned (no raising/lowering)

Binds `name` to a [`Tensor`](@ref) instance in the caller's scope and registers it in [`_TENSORS`](@ref).

### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
@def_metric g tangentM

@def_tensor T  [cotangentM, cotangentM]
@def_tensor F  [cotangentM, cotangentM] symmetries=[antisymmetric(2)]
@def_tensor R  [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()]
@def_tensor W  [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()] traceless=true print_as="Weyl"
```


In [43]:
@def_tensor T [cotangentM, cotangentM]

┌ Warning: No metric is defined on vbundle of reference :tangentM. Tensor :T has no metric assigned; indices cannot be raised or lowered.
└ @ Main ~/JuliaTensor/SymbolicTensors/src/tensors.jl:472


Defined tensor :T on manifold :M, vbundle of reference :tangentM, 2 slot(s), metric=nothing


In [44]:
T

Manifold,M
Vbundle of reference,tangentM
Rank,2
Slots,↓cotangentM ↓cotangentM
Symmetries,"SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])"
Traceless,false
Metric,none


In [45]:
(T.is_traceless,T.known_traces,T.manifold,T.metric,T.print_as,T.rank,T.slots,T.symmetries)

(false, Any[], :M, nothing, "T", 2, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])])

In [46]:
T.symmetries

1-element Vector{SlotSymmetry}:
 NoSymmetry(n=2)

In [47]:
T.slots

2-element Vector{Symbol}:
 :cotangentM
 :cotangentM

In [48]:
# Valid expressions 
(T[-a1,-a2],T[-A1,-A2])

(TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], "T", nothing, 1), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)], 0xb143a522ea70dc22), TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], "T", nothing, 1), AbstractIndex[FrameIndex(:A1, :cotangentM), FrameIndex(:A2, :cotangentM)], 0xcbed7139d2c5bdd7))

##### Error handling 

In [49]:
# Invalid expressions when no metric are present 
(T[a1,-a2],T[-a1,a2],T[a1,a2],T[-A1,-A2],T[A1,-a1],T[A1,a1])

LoadError: TensorComponent: index :a1 is on vbundle :tangentM, but slot 1 of T expects :cotangentM (tensor has no metric for raising/lowering).

In [50]:
T[a1,a1]

LoadError: TensorComponent: index :a1 is on vbundle :tangentM, but slot 1 of T expects :cotangentM (tensor has no metric for raising/lowering).

In [51]:
T.metric

#### Tensors on user defined vbundles

In [52]:
@def_vbundle E M dim [B1, B2, B3, B4]
@def_tensor K [E,dualE]

Defined VBundle E (dim=dim) and dual dualE over manifold M
Defined tensor :K on manifold :M, vbundle of reference :E, 2 slot(s), metric=nothing


┌ Warning: No metric is defined on vbundle of reference :E. Tensor :K has no metric assigned; indices cannot be raised or lowered.
└ @ Main ~/JuliaTensor/SymbolicTensors/src/tensors.jl:472


In [53]:
K[B1,-B2]

K[B1, -B2]

In [54]:
K[-a1,a2] 

LoadError: TensorComponent: index :a1 has vbundle of reference :tangentM, but tensor K has vbundle of reference :E.

### b. Kronecker delta tensor

In [55]:
@doc KroneckerDelta

```julia
KroneckerDelta
```

The package-level Kronecker delta (identity tensor) singleton.

`kronecker_delta` is the single global instance of this type. It is defined at package load time and requires no manifold or vbundle registration.

It accepts any pair of indices `[idx_up, idx_down]` at runtime, provided:

  * Both indices are of the same kind (`CoordinateIndex` or `FrameIndex`)
  * `idx_up` is contravariant: `_VBUNDLES[idx_up.vbundle].isref == true`
  * `idx_down` is covariant: `_VBUNDLES[idx_down.vbundle].isref == false`
  * They share the same vbundle of reference: `idx_up.vbundle == _VBUNDLES[idx_down.vbundle].dual`

### Usage

```julia
kronecker_delta[a1, -a2]   # identity on tangentM 
kronecker_delta[B1, -B2]   # identity on E 
kronecker_delta[-a1, a2]   # error: first index must be contravariant
kronecker_delta[a1, -B2]   # error: mixed index kinds
kronecker_delta[a1, -a1]   # valid: same symbol allowed (trace = dim)
```

### Mathematical meaning

`δ^a_b` is the identity endomorphism `id_V : V → V` for any vector space `V`. It satisfies `δ^a{}_b T^b = T^a` under contraction (future).


In [56]:
(kronecker_delta[a1,-a2],kronecker_delta[A1,-A2],kronecker_delta[a1,-a1],kronecker_delta[A1,-A1])

(TensorComponent(KroneckerDelta(), AbstractIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :cotangentM)], 0xfd3b14d6541b4010), TensorComponent(KroneckerDelta(), AbstractIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :cotangentM)], 0xf4df550e877fd668), TensorComponent(KroneckerDelta(), AbstractIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a1, :cotangentM)], 0x685b74c538b18319), TensorComponent(KroneckerDelta(), AbstractIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A1, :cotangentM)], 0x96b0cd0f294b1c6c))

In [57]:
(kronecker_delta[-a1,a2],kronecker_delta[A1,-A2],kronecker_delta[a1,-a1],kronecker_delta[A1,-A1])

LoadError: KroneckerDelta: first index :a1 must be contravariant (from a reference vbundle with isref=true). Got vbundle :cotangentM with isref=false. Did you mean kronecker_delta[a2, -a1]?

### c. Expansion of a tensor within a basis 

In [58]:
@doc basis_expansion

```julia
basis_expansion(T::Tensor, style::ExpansionStyle=Coordinate) -> BasisExpansion
basis_expansion(T::Tensor) -> BasisExpansion
```

Expand tensor `T` slot-by-slot using canonical indices and the given [`ExpansionStyle`](@ref).

  * **`Coordinate`** (default): per slot use `:coordinate` basis if registered, else `:frame`.
  * **`Frame`**: per slot use `:frame` basis only.

# Examples

```julia
basis_expansion(T)             # Coordinate style
basis_expansion(T, Coordinate)
basis_expansion(T, Frame)
```


In [59]:
basis_expansion(T)

TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], "T", nothing, 1), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)], 0xb143a522ea70dc22) BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a1, :tangentM)) ⊗ BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a2, :tangentM))

In [60]:
basis_expansion(kronecker_delta)

LoadError: MethodError: no method matching basis_expansion(::KroneckerDelta)
The function `basis_expansion` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  basis_expansion([91m::Tensor[39m)
[0m[90m   @[39m [35mSymbolicTensors[39m [90m~/JuliaTensor/SymbolicTensors/src/[39m[90m[4mframes.jl:569[24m[39m
[0m  basis_expansion([91m::Tensor[39m, [91m::ExpansionStyle[39m)
[0m[90m   @[39m [35mSymbolicTensors[39m [90m~/JuliaTensor/SymbolicTensors/src/[39m[90m[4mframes.jl:571[24m[39m


In [61]:
(basis_expansion(T, Coordinate),basis_expansion(T, Frame))

(BasisExpansion(TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], "T", nothing, 1), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)], 0xb143a522ea70dc22), BasisElement[BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a1, :tangentM)), BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a2, :tangentM))]), BasisExpansion(TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], "T", nothing, 1), AbstractIndex[FrameIndex(:A1, :cotangentM), FrameIndex(:A2, :cotangentM)], 0xcbed7139d2c5bdd7), BasisElement[BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A1, :tangentM)), BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A2, :tangentM))]))

In [62]:
@def_tensor B [tangentM, cotangentM]

Defined tensor :B on manifold :M, vbundle of reference :tangentM, 2 slot(s), metric=nothing


┌ Warning: No metric is defined on vbundle of reference :tangentM. Tensor :B has no metric assigned; indices cannot be raised or lowered.
└ @ Main ~/JuliaTensor/SymbolicTensors/src/tensors.jl:472


In [63]:
(basis_expansion(B,Coordinate),basis_expansion(B, Frame))

(BasisExpansion(TensorComponent(Tensor(:M, [:tangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], "B", nothing, 3), AbstractIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :cotangentM)], 0xf0795b074d98f3e4), BasisElement[BasisElement(Basis(:cf_M, :tangentM, :coordinate, "∂"), CoordinateIndex(:a1, :cotangentM)), BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a2, :tangentM))]), BasisExpansion(TensorComponent(Tensor(:M, [:tangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], "B", nothing, 3), AbstractIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :cotangentM)], 0xe81d9b3f80fd8a3c), BasisElement[BasisElement(Basis(:mf_M, :tangentM, :frame, "e"), FrameIndex(:A1, :cotangentM)), BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A2, :tangentM))]))

## 4. Metrics 

### a. definition

In [64]:
@doc @def_metric

```julia
@def_metric name vbundle
```

Define a new metric tensor for vbundle of reference `vbundle`, bind it to `name` in the caller's scope, and register it in both [`_TENSORS`](@ref) and [`_METRICS`](@ref).

`vbundle` must be a registered [`VBundle`](@ref) with `isref == true` (the bundle named in [`@def_manifold`](@ref) or [`@def_vbundle`](@ref)).

`@def_metric` is a specialised shortcut (not a call to [`@def_tensor`](@ref)) that always builds a rank-2 fully covariant symmetric metric:

  * Slots: `[dual(vbundle), dual(vbundle)]` (both covariant).
  * `symmetries=[symmetric(2)]` — fixed, no user override.
  * `print_as` set to `string(name)`; `metric` set to `name` (self-referential).
  * `_METRICS[name] = vbundle` (vbundle of reference).
  * No keyword arguments.

At **expression** time you still write `g[-a1, -a2]` using coordinate indices; only **definition** uses `@def_metric g tangentM`.

# Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]

@def_metric g tangentM    # Riemannian metric on M
@def_metric η tangentM    # second metric, same vbundle of reference
```


In [65]:
@def_metric g tangentM

Defined metric g on vbundle of reference tangentM (manifold :M)


In [66]:
(g[-a1,-a2],g[-A1,-A2])

(TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], 1)], SignedPerm[SignedPerm([2, 1], 1), SignedPerm([1, 2], 1)])], false, Any[], "g", :g, 4), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)], 0x95970ac3ff31d653), TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], 1)], SignedPerm[SignedPerm([2, 1], 1), SignedPerm([1, 2], 1)])], false, Any[], "g", :g, 4), AbstractIndex[FrameIndex(:A1, :cotangentM), FrameIndex(:A2, :cotangentM)], 0xb040d6dae786b808))

In [67]:
@def_metric eta E

Defined metric eta on vbundle of reference E (manifold :M)


In [68]:
(basis_expansion(g),basis_expansion(g,Frame))

(BasisExpansion(TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], 1)], SignedPerm[SignedPerm([2, 1], 1), SignedPerm([1, 2], 1)])], false, Any[], "g", :g, 4), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)], 0x95970ac3ff31d653), BasisElement[BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a1, :tangentM)), BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a2, :tangentM))]), BasisExpansion(TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], 1)], SignedPerm[SignedPerm([2, 1], 1), SignedPerm([1, 2], 1)])], false, Any[], "g", :g, 4), AbstractIndex[FrameIndex(:A1, :cotangentM), FrameIndex(:A2, :cotangentM)], 0xb040d6dae786b808), BasisElement[BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A1, :tangentM)), BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A2, :tangentM))]))

### b. Error handling 

In [69]:
@def_metric g cotangentM

LoadError: @def_metric: vbundle cotangentM is not a vbundle of reference (isref must be true). Pass the ref vbundle, e.g. tangentM not cotangentM.

### Tensors again (with a metric) 

In [70]:
? @def_tensor

```julia
@def_tensor name [vbundle1, vbundle2, ...] [symmetries=...] [traceless=...] [print_as="..."] [metric=...]
```

Define a new abstract tensor and bind it to `name` in the caller's scope.

### Slot syntax

Specify slots directly as VBundle symbols. The manifold is automatically deduced from the first VBundle's `manifold` field.

  * `:tangentM` → contravariant (upper) slot
  * `:cotangentM` → covariant (lower) slot
  * Any user-defined VBundle from `@def_vbundle`

All slots must derive from the **same vbundle of reference**. A vbundle of reference is any vbundle `v` with `v.isref = true`. For a slot `vb`: if `vb.isref` is `true`, its vbundle of reference is `vb`; if `false`, its vbundle of reference is `vb`'s dual. Mixing slots from distinct vbundles of reference is an error — such objects are not tensors in the strict mathematical sense.

### Registry collision rules

1. Same name, same vbundle of reference, same slots → **warn and redefine**
2. Same name, same vbundle of reference, different slots → **error**
3. Same name, different vbundle of reference → **error**

Rule 3 ensures that a variable name in scope maps to exactly one mathematical object with no ambiguity.

### Keyword arguments (all optional)

  * `symmetries`  : a [`SlotSymmetry`](@ref) or `Vector{SlotSymmetry}` describing                 the slot permutation symmetry group(s). Defaults to                 `[no_symmetry(rank)]`.
  * `traceless`   : `true` if any self-contraction of this tensor is zero                 (e.g. Weyl tensor). Defaults to `false`.
  * `print_as`    : display label string. Defaults to `string(name)`.                 Example: `print_as="R"` or `print_as=:R` (symbol sugar).
  * `metric`      : name of the metric tensor to associate with this tensor.                 Omitting this keyword triggers automatic resolution on the                 tensor's vbundle of reference:                 - one metric on that vbundle → assigned silently                 - multiple metrics → `@warn`, first sorted name is assigned                 - no metric → `@warn`, `nothing` assigned (no raising/lowering)

Binds `name` to a [`Tensor`](@ref) instance in the caller's scope and registers it in [`_TENSORS`](@ref).

### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
@def_metric g tangentM

@def_tensor T  [cotangentM, cotangentM]
@def_tensor F  [cotangentM, cotangentM] symmetries=[antisymmetric(2)]
@def_tensor R  [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()]
@def_tensor W  [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()] traceless=true print_as="Weyl"
```


In [71]:
@def_tensor F  [cotangentM, cotangentM] symmetries=[antisymmetric(2)]

Defined tensor :F on manifold :M, vbundle of reference :tangentM, 2 slot(s), metric=g


In [72]:
(F[-a1,-a2],F[a1,-a2],F[-a1,a2],F[a1,a2],F[-A1,-A2],F[A1,-A2],F[-A1,A2],F[A1,A2])

(TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], -1)], SignedPerm[SignedPerm([2, 1], -1), SignedPerm([1, 2], 1)])], false, Any[], "F", :g, 6), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)], 0x092373ffcb8b1b2f), TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], -1)], SignedPerm[SignedPerm([2, 1], -1), SignedPerm([1, 2], 1)])], false, Any[], "F", :g, 6), AbstractIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :cotangentM)], 0x26f41fb0e298ab8a), TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], -1)], SignedPerm[SignedPerm([2, 1], -1), SignedPerm([1, 2], 1)])], false, Any[], "F", :g, 6), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :tangentM)], 0x52957a4211f85a8b), TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], -1)], SignedPerm[SignedPerm([2, 1], -1), SignedPerm([1, 2], 1)])], false, Any[], "F", :g, 6), AbstractIndex[CoordinateIndex(:a1, :tangentM), CoordinateIndex(:a2, :tangentM)], 0xb6864426796a8e7e), TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], -1)], SignedPerm[SignedPerm([2, 1], -1), SignedPerm([1, 2], 1)])], false, Any[], "F", :g, 6), AbstractIndex[FrameIndex(:A1, :cotangentM), FrameIndex(:A2, :cotangentM)], 0x23cd4016b3dffce4), TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], -1)], SignedPerm[SignedPerm([2, 1], -1), SignedPerm([1, 2], 1)])], false, Any[], "F", :g, 6), AbstractIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :cotangentM)], 0x1e985fe915fd41e2), TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], -1)], SignedPerm[SignedPerm([2, 1], -1), SignedPerm([1, 2], 1)])], false, Any[], "F", :g, 6), AbstractIndex[FrameIndex(:A1, :cotangentM), FrameIndex(:A2, :tangentM)], 0xa22032b517dd4194), TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], -1)], SignedPerm[SignedPerm([2, 1], -1), SignedPerm([1, 2], 1)])], false, Any[], "F", :g, 6), AbstractIndex[FrameIndex(:A1, :tangentM), FrameIndex(:A2, :tangentM)], 0xff9821e664ce43d3))

In [73]:
F[-B1,B2]

LoadError: TensorComponent: index :B1 has vbundle of reference :E, but tensor F has vbundle of reference :tangentM.

In [74]:
@def_tensor T  [cotangentM, cotangentM]

Defined tensor :T on manifold :M, vbundle of reference :tangentM, 2 slot(s), metric=g


┌ Warning: Tensor :T is already defined with the same slots. Redefining.
└ @ Main ~/JuliaTensor/SymbolicTensors/src/tensors.jl:425


In [75]:
T

Manifold,M
Vbundle of reference,tangentM
Rank,2
Slots,↓cotangentM ↓cotangentM
Symmetries,"SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])"
Traceless,false
Metric,g


In [76]:
(basis_expansion(T,Coordinate),basis_expansion(T,Frame))

(BasisExpansion(TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], "T", :g, 7), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)], 0xfb42769244766ef6), BasisElement[BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a1, :tangentM)), BasisElement(Basis(:ccf_M, :cotangentM, :coordinate, "dx"), CoordinateIndex(:a2, :tangentM))]), BasisExpansion(TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], "T", :g, 7), AbstractIndex[FrameIndex(:A1, :cotangentM), FrameIndex(:A2, :cotangentM)], 0x15ec42a92ccb50ab), BasisElement[BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A1, :tangentM)), BasisElement(Basis(:mcf_M, :cotangentM, :frame, "θ"), FrameIndex(:A2, :tangentM))]))

## 5. Tensor algebra 

In [77]:
2*F[-a1,-a2]+ T[-a1,-a2]+3*T[-a1,-a2]

2 * F[-a1, -a2] + 4 * T[-a1, -a2]

In [78]:
repr(T[-a1,-a2])

"TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], \"T\", :g, 7), AbstractIndex[CoordinateIndex(:a1, :cotangentM), CoordinateIndex(:a2, :cotangentM)], 0xfb42769244766ef6)"

In [79]:
F[-a1,-a2]+ T[-a1,-a2]+3*T[-a1,-a2]

F[-a1, -a2] + 4 * T[-a1, -a2]

In [80]:
F[-a1,-a2]-1/2 g[-a1,-a2]*F[b1,-b1]==0

LoadError: MethodError: no method matching -(::TensorComponent, ::Float64)
The function `-` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  -([91m::Complex{Bool}[39m, ::Real)
[0m[90m   @[39m [90mBase[39m [90m[4mcomplex.jl:329[24m[39m
[0m  -(::TensorComponent)
[0m[90m   @[39m [35mSymbolicTensors[39m [90m~/JuliaTensor/SymbolicTensors/src/[39m[90m[4mtensorComponentExpr.jl:355[24m[39m
[0m  -(::TensorComponent, [91m::TensorComponent[39m)
[0m[90m   @[39m [35mSymbolicTensors[39m [90m~/JuliaTensor/SymbolicTensors/src/[39m[90m[4mtensorComponentExpr.jl:361[24m[39m
[0m  ...


In [81]:
g[-a3,-a4]*(F[-a1,-a2]+ T[-a1,-a2]+3*T[-a1,-a2])

g[-a3, -a4] * F[-a1, -a2] + 4 * g[-a3, -a4] * T[-a1, -a2]

In [82]:
F[-a1,-a2]+ T[-a1,-a2]+3*T[-a1,-a2]

F[-a1, -a2] + 4 * T[-a1, -a2]

In [83]:
F[-a1,-a2]+ T[a1,-a2]+3*T[-a1,-a2]

F[-a1, -a2] + 3 * T[-a1, -a2] + T[a1, -a2]

In [84]:
@def_tensor H  [cotangentM, cotangentM]

Defined tensor :H on manifold :M, vbundle of reference :tangentM, 2 slot(s), metric=g


In [85]:
?@add_indices

```julia
@add_indices M idx1 idx2 ...
```

Register extra **coordinate** index symbols to the tangent bundle of manifold `M` and bind each to a contravariant [`CoordinateIndex`](@ref) in scope.

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
@add_indices M a5 a6
a5   # CoordinateIndex(:a5, :tangentM)
```


In [86]:
@add_indices M a5 a6

In [87]:
(g[-a3,-a4]*H[-a5,-a6])*(F[-a1,-a2]+ T[-a1,-a2]+3*T[-a1,-a2])

g[-a3, -a4] * F[-a1, -a2] * H[-a5, -a6] + 4 * g[-a3, -a4] * T[-a1, -a2] * H[-a5, -a6]

In [88]:
-g[-a3,-a4]*H[-a5,-a6]

-g[-a3, -a4] * H[-a5, -a6]

In [89]:
g[-a3,-a4]*H[-a5,-a6]+H[-a5,-a6]*g[-a3,-a4]

2 * g[-a3, -a4] * H[-a5, -a6]

In [90]:
_TENSORS

Dict{Symbol, Tensor} with 7 entries:
  :T   => Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, S…
  :F   => Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, S…
  :H   => Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, S…
  :eta => Tensor(:M, [:dualE, :dualE], SlotSymmetry[SlotSymmetry(2, SignedPerm[…
  :K   => Tensor(:M, [:E, :dualE], SlotSymmetry[SlotSymmetry(2, SignedPerm[], S…
  :B   => Tensor(:M, [:tangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, Sig…
  :g   => Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, S…

In [91]:
repr(_TENSORS)

"Dict{Symbol, Tensor}(:T => Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], \"T\", :g, 7), :F => Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([" ⋯ 668 bytes ⋯ "rm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], \"B\", nothing, 3), :g => Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], 1)], SignedPerm[SignedPerm([2, 1], 1), SignedPerm([1, 2], 1)])], false, Any[], \"g\", :g, 4))"

In [92]:
_COORDINATE_INDICES

Dict{Symbol, Symbol} with 6 entries:
  :a5 => :tangentM
  :a2 => :tangentM
  :a3 => :tangentM
  :a6 => :tangentM
  :a4 => :tangentM
  :a1 => :tangentM

In [93]:
_FRAME_INDICES

Dict{Symbol, Symbol} with 8 entries:
  :A1 => :tangentM
  :B3 => :E
  :A3 => :tangentM
  :B4 => :E
  :B1 => :E
  :A2 => :tangentM
  :A4 => :tangentM
  :B2 => :E

In [94]:
@add_indices M a7 a8

In [95]:
components = [
        g[a1, -a2],
        H[a3, -a4],
        F[a5, -a6],
        T[a7, -a8],
    ]

4-element Vector{TensorComponent}:
 g[a1, -a2]
 H[a3, -a4]
 F[a5, -a6]
 T[a7, -a8]

In [96]:
p = components[1] * components[2] * components[3] * components[4]
isconcretetype(typeof(p))

true

In [97]:
repr(p)

"TensorComponentProduct(TensorComponent[TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[SignedPerm([2, 1], 1)], SignedPerm[SignedPerm([2, 1], 1), SignedPerm([1, 2], 1)])], false, Any[], \"g\", :g, 4), AbstractIndex[CoordinateI" ⋯ 653 bytes ⋯ "8), TensorComponent(Tensor(:M, [:cotangentM, :cotangentM], SlotSymmetry[SlotSymmetry(2, SignedPerm[], SignedPerm[SignedPerm([1, 2], 1)])], false, Any[], \"H\", :g, 8), AbstractIndex[CoordinateIndex(:a3, :tangentM), CoordinateIndex(:a4, :cotangentM)], 0x3ffa5cf8440916ff)])"

In [98]:
@code_warntype hash(components[1], UInt(0))

MethodInstance for hash(::TensorComponent, ::UInt64)
  from hash(e::TensorComponent, h::UInt64) @ SymbolicTensors ~/JuliaTensor/SymbolicTensors/src/tensorComponents.jl:282
Arguments
  #self#::Core.Const(hash)
  e::TensorComponent
  h::UInt64
Body::UInt64
1 ─ %1 = SymbolicTensors.xor::Core.Const(xor)
│   %2 = Base.getproperty(e, :_hash)::UInt64
│   %3 = (%1)(%2, h)::UInt64
└──      return %3



In [99]:
(isbitstype(TensorComponentProduct),isbitstype(TensorComponent))


(false, false)

In [100]:
using Profile, PProf, Random

In [101]:
Random.seed!(0xC0FFEE)
random_terms = map(1:1000) do i
    shuffled = shuffle(components)
    prod = shuffled[1] * shuffled[2] * shuffled[3] * shuffled[4]
    return term(prod)
end

1000-element Vector{TensorComponentTerm{Int64, TensorComponentProduct}}:
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 ⋮
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]
 g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]

In [102]:
@code_warntype TensorComponentSum(random_terms)

MethodInstance for TensorComponentSum(::Vector{TensorComponentTerm{Int64, TensorComponentProduct}})
  from TensorComponentSum(raw_terms::AbstractVector{T}) where T<:TensorComponentTerm @ SymbolicTensors ~/JuliaTensor/SymbolicTensors/src/tensorComponentExpr.jl:182
Static Parameters
  T = TensorComponentTerm{Int64, TensorComponentProduct}
Arguments
  #ctor-self#::Type{TensorComponentSum}
  raw_terms::Vector{TensorComponentTerm{Int64, TensorComponentProduct}}
Locals
  final_terms::Vector{TensorComponentTerm{Int64, TensorComponentProduct}}
  @_4::Vector{TensorComponentTerm{Int64, TensorComponentProduct}}
  @_5::Vector{TensorComponentTerm{Int64, TensorComponentProduct}}
  @_6::Vector{TensorComponentTerm{Int64, TensorComponentProduct}}
Body::TensorComponentSum{TensorComponentTerm{Int64, TensorComponentProduct}}
1 ──       Core.NewvarNode(:(final_terms))
│    %2  = SymbolicTensors.isempty::Core.Const(isempty)
│    %3  = (%2)(raw_terms)::Bool
└───       goto #7 if not %3
2 ── %5  = SymbolicTen

In [103]:
raw = Vector{TensorComponentTerm}(random_terms)

# Warmup
SymbolicTensors._merge_terms(raw)
@time SymbolicTensors._merge_terms(raw)

# Warmup
begin
    merged = Dict{TensorComponentProduct, Int}()
    sizehint!(merged, 1000)
    for t in raw
        b = t.body
        if haskey(merged, b); merged[b] += t.coeff
        else; merged[b] = t.coeff
        end
    end
end
@time begin
    merged = Dict{TensorComponentProduct, Int}()
    sizehint!(merged, 1000)
    for t in raw
        b = t.body
        if haskey(merged, b); merged[b] += t.coeff
        else; merged[b] = t.coeff
        end
    end
end

  0.000204 seconds (1.50 k allocations: 31.305 KiB)
  0.000370 seconds (6.97 k allocations: 205.820 KiB)


In [104]:
raw = Vector{TensorComponentTerm}(random_terms)
SymbolicTensors._merge_terms(raw)  # warmup
@time SymbolicTensors._merge_terms(raw)

  0.000257 seconds (1.50 k allocations: 31.305 KiB)


1-element Vector{TensorComponentTerm}:
 1000 * g[a1, -a2] * F[a5, -a6] * T[a7, -a8] * H[a3, -a4]

In [105]:
# Isolate _merge_terms alone
raw = random_terms  # already Vector{TensorComponentTerm{Int64, TensorComponentProduct}}
SymbolicTensors._merge_terms(raw)  # warmup
@allocated SymbolicTensors._merge_terms(raw)

16264

In [106]:
merged = Dict{TensorComponentProduct, Int64}()
sizehint!(merged, 1000)
for t in raw
    b = t.body
    if haskey(merged, b)
        merged[b] = merged[b] + t.coeff
    else
        merged[b] = t.coeff
    end
end
@allocated begin
    merged2 = Dict{TensorComponentProduct, Int64}()
    sizehint!(merged2, 1000)
    for t in raw
        b = t.body
        if haskey(merged2, b)
            merged2[b] = merged2[b] + t.coeff
        else
            merged2[b] = t.coeff
        end
    end
end

202584

In [107]:
@code_warntype SymbolicTensors._merge_terms(raw)

MethodInstance for SymbolicTensors._merge_terms(::Vector{TensorComponentTerm{Int64, TensorComponentProduct}})
  from _merge_terms(raw_terms::AbstractVector{T}) where T<:TensorComponentTerm @ SymbolicTensors ~/JuliaTensor/SymbolicTensors/src/tensorComponentExpr.jl:268
Static Parameters
  T = TensorComponentTerm{Int64, TensorComponentProduct}
Arguments
  #self#::Core.Const(SymbolicTensors._merge_terms)
  raw_terms::Vector{TensorComponentTerm{Int64, TensorComponentProduct}}
Locals
  @_3::Union{Nothing, Tuple{Int64, Int64}}
  val::Nothing
  #49::SymbolicTensors.var"#_merge_terms##0#_merge_terms##1"
  curr_coeff::Int64
  curr_body::TensorComponentProduct
  out::Vector{TensorComponentTerm{Int64, TensorComponentProduct}}
  sorted::Vector{TensorComponentTerm{Int64, TensorComponentProduct}}
  i::Int64
  t::TensorComponentTerm{Int64, TensorComponentProduct}
Body::Vector{TensorComponentTerm{Int64, TensorComponentProduct}}
1 ──        Core.NewvarNode(:(@_3))
│           Core.NewvarNode(:(val))
│  